Language: **English (default)** | [Polski](examples.ipynb)

# Lecture 11 -- experiment: from a probability distribution to text generation

Companion material to [`lecture.en.md`](lecture.en.md). Five experiments:
(A) tokenization; (B) next-token prediction, logits, manual generation vs.
`model.generate()`; (C) decoding strategies; (D) chat templates; (E) KV
cache and cost.

**Note: this notebook downloads the model weights (about 257MB) on first
run** -- it requires internet access. Subsequent runs use the local
Hugging Face cache.

In [1]:
import time

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Using device: {device}")

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"  # pinned, not "the latest version"

/Users/michalmaj/Documents/python/deep-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## A. Tokenization

The same tokenizer, five very different text examples.

In [2]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
print(f"tokenizer load: {time.time() - t0:.2f}s")

tokenizer load: 0.58s


In [3]:
examples = {
    "English sentence": "The quick brown fox jumps over the lazy dog.",
    "Polish sentence": "Szybki brązowy lis przeskakuje nad leniwym psem.",
    "numbers/dates": "The invoice total is 1234.56 USD, due on 2026-03-15.",
    "special characters": "Email me at test@example.com -- see §4.2 for *details* (urgent!!).",
    "rare words": (
        "The spectrophotometric quokka antidisestablishmentarianism "
        "supercalifragilisticexpialidocious."
    ),
}

print(f"{'example':20s} {'chars':>6s} {'words':>6s} {'tokens':>7s} {'words/token':>12s}")
for name, text in examples.items():
    ids = tokenizer(text)["input_ids"]
    n_chars, n_words = len(text), len(text.split())
    print(f"{name:20s} {n_chars:>6d} {n_words:>6d} {len(ids):>7d} {n_words/len(ids):>12.2f}")

example               chars  words  tokens  words/token
English sentence         44      9      10         0.90
Polish sentence          48      7      25         0.28
numbers/dates            52      9      28         0.32
special characters       66     10      23         0.43
rare words               94      5      25         0.20


In [4]:
# Let's look at the actual tokens for the two most contrasting examples
for name in ["English sentence", "Polish sentence"]:
    text = examples[name]
    ids = tokenizer(text)["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(f"--- {name} ---")
    print(f"text: {text!r}")
    print(f"tokens ({len(tokens)}): {tokens}")
    print(f"id: {ids}")
    print()

--- English sentence ---
text: 'The quick brown fox jumps over the lazy dog.'
tokens (10): ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.']
id: [504, 2365, 6354, 16438, 27003, 690, 260, 23790, 2767, 30]

--- Polish sentence ---
text: 'Szybki brązowy lis przeskakuje nad leniwym psem.'
tokens (25): ['S', 'zy', 'b', 'ki', 'Ġbr', 'Ä', 'ħ', 'z', 'ow', 'y', 'Ġl', 'is', 'Ġpr', 'zes', 'k', 'aku', 'je', 'Ġn', 'ad', 'Ġlen', 'iw', 'ym', 'Ġp', 'sem', '.']
id: [67, 5346, 82, 8139, 1156, 143, 223, 106, 323, 105, 303, 271, 694, 16244, 91, 28277, 16864, 304, 344, 3858, 17493, 1905, 273, 8188, 30]



**Observation.** English sentence: about 1 token per word. Polish
sentence: about **3-4 tokens per word** -- the same tokenizer,
dramatically different granularity, because it was trained mostly on
English text/code. **You cannot assume a fixed word:token ratio,
especially across languages.**

Note: the Polish sentence above ("Szybki brązowy lis przeskakuje nad
leniwym psem." -- "A quick brown fox jumps over a lazy dog.") is kept in
its original Polish on purpose -- the entire point of this comparison is
how the *same English-trained tokenizer* handles English vs. Polish text,
so translating it would remove the very thing being measured.

In [5]:
# numbers and special characters - details
for name in ["numbers/dates", "special characters", "rare words"]:
    text = examples[name]
    tokens = tokenizer.convert_ids_to_tokens(tokenizer(text)["input_ids"])
    print(f"--- {name} ---")
    print(f"text: {text!r}")
    print(f"tokens: {tokens}")
    print()

print(
    "Observations: digits are often split INDIVIDUALLY (each digit is its own token); "
    "special characters outside ASCII can give oddly-looking tokens (byte-level BPE "
    "encoding); rare/long words are split into sensible subwords, NOT replaced by a "
    "single '<UNK>' token - that is exactly the advantage of subword tokenization."
)

--- numbers/dates ---
text: 'The invoice total is 1234.56 USD, due on 2026-03-15.'
tokens: ['The', 'Ġinvoice', 'Ġtotal', 'Ġis', 'Ġ', '1', '2', '3', '4', '.', '5', '6', 'ĠUSD', ',', 'Ġdue', 'Ġon', 'Ġ', '2', '0', '2', '6', '-', '0', '3', '-', '1', '5', '.']

--- special characters ---
text: 'Email me at test@example.com -- see §4.2 for *details* (urgent!!).'
tokens: ['Email', 'Ġme', 'Ġat', 'Ġtest', '@', 'example', '.', 'com', 'Ġ--', 'Ġsee', 'ĠÂ§', '4', '.', '2', 'Ġfor', 'Ġ*', 'details', '*', 'Ġ(', 'urg', 'ent', '!!', ').']

--- rare words ---
text: 'The spectrophotometric quokka antidisestablishmentarianism supercalifragilisticexpialidocious.'
tokens: ['The', 'Ġspect', 'roph', 'ot', 'ometric', 'Ġqu', 'ok', 'ka', 'Ġant', 'idis', 'establish', 'ment', 'arianism', 'Ġsuper', 'cal', 'if', 'rag', 'il', 'istice', 'x', 'p', 'ial', 'id', 'ocious', '.']

Observations: digits are often split INDIVIDUALLY (each digit is its own token); special characters outside ASCII can give oddly-looking tokens (b

## B. Next-token prediction

We load the model. For a short prompt we take the last position's
logits, compute the distribution, look at the effect of temperature,
generate manually step by step, and compare with `model.generate()`.

In [6]:
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float32)
model.eval()
print(f"model load: {time.time() - t0:.2f}s")
print(f"number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model.config)

W1019 14:18:44.385000 86998 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/272 [00:00<00:55,  4.87it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 972.49it/s]

model load: 0.49s
number of parameters: 134,515,008
LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "float32",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 576,
  "initializer_range": 0.041666666666666664,
  "intermediate_size": 1536,
  "is_llama_config": true,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 9,
  "num_hidden_layers": 30,
  "num_key_value_heads": 3,
  "pad_token_id": 2,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_interleaved": false,
  "rope_parameters": {
    "rope_theta": 100000,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers.js_config": {
    "kv_cache_dtype": {
      "fp16": "float16",
      "q4f16": "float16"
    }
  },
  "transformers_version": "5.20.1",
  "use_cache": true,
  "vocab_size": 49152
}



In [7]:
messages = [{"role": "user", "content": "The capital of France is"}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
)
prompt_len = inputs["input_ids"].shape[1]
print(f"prompt (after chat template): {prompt_len} tokens")
print(tokenizer.decode(inputs["input_ids"][0]))

prompt (after chat template): 35 tokens
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
The capital of France is<|im_end|>
<|im_start|>assistant



In [8]:
with torch.no_grad():
    out = model(**inputs)
next_token_logits = out.logits[0, -1, :]
print(f"logits shape: {next_token_logits.shape}")

probs = F.softmax(next_token_logits, dim=-1)
topk_probs, topk_ids = probs.topk(5)
print("\ntop 5 most probable next tokens:")
for p, idx in zip(topk_probs.tolist(), topk_ids.tolist()):
    print(f"  id={idx:>6d}  token={tokenizer.decode([idx])!r:12s}  p={p:.4f}")

logits shape: torch.Size([49152])

top 5 most probable next tokens:
  id=   504  token='The'         p=0.3113
  id=    50  token='B'           p=0.0674
  id=    18  token='"'           p=0.0616
  id= 30904  token='Bon'         p=0.0353
  id= 38634  token='Paris'       p=0.0344


In [9]:
print("Effect of temperature on the same logit distribution:")
for tau in [0.5, 1.0, 2.0]:
    p_tau = F.softmax(next_token_logits / tau, dim=-1)
    top3 = p_tau.topk(3)
    pairs = zip(top3.values.tolist(), top3.indices.tolist())
    tokens_str = [f"{tokenizer.decode([i])!r}:{v:.3f}" for v, i in pairs]
    print(f"  tau={tau}: top-3 {tokens_str}")

print(
    "\nSmaller tau -> greater concentration on the already-most-probable token; "
    "larger tau -> flatter distribution. This is the SAME logits, a different transformation."
)

Effect of temperature on the same logit distribution:
  tau=0.5: top-3 ["'The':0.866", "'B':0.041", '\'"\':0.034']
  tau=1.0: top-3 ["'The':0.311", "'B':0.067", '\'"\':0.062']
  tau=2.0: top-3 ["'The':0.026", "'B':0.012", '\'"\':0.012']

Smaller tau -> greater concentration on the already-most-probable token; larger tau -> flatter distribution. This is the SAME logits, a different transformation.


In [10]:
# manual step-by-step (greedy) generation
n_steps = 10
manual_ids = inputs["input_ids"].clone()
manual_tokens = []
with torch.no_grad():
    for step in range(n_steps):
        step_out = model(manual_ids)
        next_id = step_out.logits[0, -1, :].argmax().item()
        manual_tokens.append(next_id)
        manual_ids = torch.cat([manual_ids, torch.tensor([[next_id]])], dim=1)
        if next_id == tokenizer.eos_token_id:
            print(f"EOS generated at step {step} - stopping")
            break

manual_text = tokenizer.decode(manual_tokens, skip_special_tokens=True)
print(f"manual tokens: {manual_tokens}")
print(f"manual text: {manual_text!r}")

EOS generated at step 7 - stopping
manual tokens: [504, 3575, 282, 4649, 314, 7042, 30, 2]
manual text: 'The capital of France is Paris.'


In [11]:
# model.generate() with matching settings - the SAME prompt (no formatting change)
with torch.no_grad():
    gen_out = model.generate(
        input_ids=inputs["input_ids"], max_new_tokens=n_steps, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
generate_tokens = gen_out[0, prompt_len:].tolist()
generate_text = tokenizer.decode(generate_tokens, skip_special_tokens=True)
print(f"generate() tokens: {generate_tokens}")
print(f"generate() text: {generate_text!r}")

match = manual_tokens == generate_tokens[: len(manual_tokens)]
print(f"\nmanual tokens == generate() tokens: {match}")
assert match
print(
    "Agreement confirmed: 'generate()' with do_sample=False is (mathematically) "
    "exactly the same loop as manually calling model() + argmax + concatenation."
)

generate() tokens: [504, 3575, 282, 4649, 314, 7042, 30, 2]
generate() text: 'The capital of France is Paris.'

manual tokens == generate() tokens: True
Agreement confirmed: 'generate()' with do_sample=False is (mathematically) exactly the same loop as manually calling model() + argmax + concatenation.


### `pipeline()` as a convenient abstraction (briefly)

`transformers.pipeline` hides tokenization, `generate()`, and decoding
behind one call -- convenient in practice, but we **do not** use it as
the main tool in this notebook, because it hides exactly the stages we
want to see.

In [12]:
from transformers import pipeline

# pipeline() auto-detects an accelerator (e.g. MPS on Mac) by default - explicit
# device=device keeps it consistent with the rest of the notebook (CPU, see section 0)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)
pipe_out = pipe(messages, max_new_tokens=10, do_sample=False)
pipe_text = pipe_out[0]["generated_text"][-1]["content"]
print(f"pipeline() text: {pipe_text!r}")
print(f"manual/generate() text (from above): {manual_text!r}")
print(
    "The same result, three lines of code - but 'what exactly happens' is hidden. "
    "In this module we use generate() as the main tool precisely because the stages "
    "(tokenization, logits, token selection, decoding) need to stay visible."
)

pipeline() text: 'The capital of France is Paris.'
manual/generate() text (from above): 'The capital of France is Paris.'
The same result, three lines of code - but 'what exactly happens' is hidden. In this module we use generate() as the main tool precisely because the stages (tokenization, logits, token selection, decoding) need to stay visible.


## C. Decoding strategies

We compare greedy, sampling (different temperatures), top-k, top-p on
the same prompts, with the same length budget and controlled random
state.

In [13]:
decoding_prompt = [{"role": "user", "content":
    "Tell me a short, imaginative story about a robot who discovers music."}]
dec_inputs = tokenizer.apply_chat_template(
    decoding_prompt, tokenize=True, add_generation_prompt=True,
    return_tensors="pt", return_dict=True,
)
dec_prompt_len = dec_inputs["input_ids"].shape[1]

configs = [
    ("greedy", dict(do_sample=False)),
    ("sampling, tau=0.7", dict(do_sample=True, temperature=0.7)),
    ("sampling, tau=1.5", dict(do_sample=True, temperature=1.5)),
    ("top-k=10", dict(do_sample=True, temperature=1.0, top_k=10)),
    ("top-p=0.9", dict(do_sample=True, temperature=1.0, top_k=0, top_p=0.9)),
]

print("=== open-ended, creative prompt ===")
for name, kwargs in configs:
    torch.manual_seed(0)
    with torch.no_grad():
        out = model.generate(**dec_inputs, max_new_tokens=30,
                              pad_token_id=tokenizer.pad_token_id, **kwargs)
    text = tokenizer.decode(out[0, dec_prompt_len:], skip_special_tokens=True)
    print(f"[{name}]\n  {text!r}\n")

=== open-ended, creative prompt ===


[greedy]
  'In the heart of a bustling city, a young robot named Kael stumbled upon a hidden underground music scene. The sounds of a thousand instruments filled the'



[sampling, tau=0.7]
  "The music of the gods. So popular, so enchanted, it's like a whisper in the ear of a thousand gods. One day, a"



[sampling, tau=1.5]
  'The life of a robot seems simple but never truly ends, and not all robots thrive without exploring their true passions after they have begun their digital escapades'



[top-k=10]
  "The music of the robot. It's the promise of limitless creative possibilities, the symphony of human imagination, and the whispered promises of a new era in"



[top-p=0.9]
  'The music of the gods. So popular, so enchanted. A young princess named Lyra (heavily piped her as Lily, Lily'



**Diagnostic task for the student (markdown, not code)**: notice that
`top-k=0` in the `top-p=0.9` configuration is deliberate -- `top_k` is
**not** disabled by default, so if we did not zero it, both filters would
act **simultaneously**, and the result would not isolate the effect of
top-p alone. This is exactly "avoid accidentally using settings ignored
in a given mode" from the Experiment C section in `lecture.en.md`.

In [14]:
print("=== concentrated, factual prompt (for comparison) ===")
factual_prompt = [{"role": "user", "content": "What is the capital of France?"}]
fact_inputs = tokenizer.apply_chat_template(
    factual_prompt, tokenize=True, add_generation_prompt=True,
    return_tensors="pt", return_dict=True,
)
fact_prompt_len = fact_inputs["input_ids"].shape[1]

for name, kwargs in configs:
    torch.manual_seed(0)
    with torch.no_grad():
        out = model.generate(**fact_inputs, max_new_tokens=15,
                              pad_token_id=tokenizer.pad_token_id, **kwargs)
    text = tokenizer.decode(out[0, fact_prompt_len:], skip_special_tokens=True)
    print(f"[{name}] {text!r}")

print(
    "\nThe same set of decoding strategies, but on a factual question - the "
    "distribution is much more concentrated (the model is 'confident' in its answer), "
    "so the DIVERSITY between strategies is much smaller here than on the open-ended "
    "prompt above. Greater diversity (above) does NOT automatically mean better "
    "quality - it is a property of the distribution for a given prompt, not a "
    "universal advantage of sampling."
)

=== concentrated, factual prompt (for comparison) ===
[greedy] 'The capital of France is Paris.'


[sampling, tau=0.7] 'The capital of France is Paris. It is a city known for its rich'
[sampling, tau=1.5] 'The capital of France, and known as the City of Learn and Explore,'


[top-k=10] 'The capital of France is Paris. It is located in the northern part of'
[top-p=0.9] 'The capital of France is Paris. It is located in the northern part of'

The same set of decoding strategies, but on a factual question - the distribution is much more concentrated (the model is 'confident' in its answer), so the DIVERSITY between strategies is much smaller here than on the open-ended prompt above. Greater diversity (above) does NOT automatically mean better quality - it is a property of the distribution for a given prompt, not a universal advantage of sampling.


In [15]:
# reproducibility of sampling with a fixed seed
print("=== reproducibility with a fixed seed ===")
for trial in range(2):
    torch.manual_seed(42)
    with torch.no_grad():
        out = model.generate(**dec_inputs, max_new_tokens=15, do_sample=True, temperature=0.8,
                              top_k=50, pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0, dec_prompt_len:], skip_special_tokens=True)
    print(f"  trial {trial} (seed=42): {text!r}")
print("Identical on both runs - a fixed seed gives reproducibility, despite sampling.")

=== reproducibility with a fixed seed ===


  trial 0 (seed=42): 'One day, in a world where machines can play the piano, a robot'


  trial 1 (seed=42): 'One day, in a world where machines can play the piano, a robot'
Identical on both runs - a fixed seed gives reproducibility, despite sampling.


## D. Context and chat templates

In [16]:
plain_prompt = "What is the capital of France?"
plain_ids = tokenizer(plain_prompt, return_tensors="pt")
print(f"plain text (no chat template): {plain_ids['input_ids'].shape[1]} tokens")
print(tokenizer.decode(plain_ids["input_ids"][0]))

chat_messages = [{"role": "user", "content": plain_prompt}]
chat_text = tokenizer.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)
print(f"\nwith chat template:\n{chat_text!r}")
# return_dict=True: apply_chat_template with tokenize=True returns a BatchEncoding
# (a dict of input_ids/attention_mask), NOT a bare tensor - explicit return_dict=True
# makes this visible in the code, instead of assuming the result's shape
chat_inputs = tokenizer.apply_chat_template(
    chat_messages, tokenize=True, add_generation_prompt=True,
    return_tensors="pt", return_dict=True,
)
print(f"\nwith chat template: {chat_inputs['input_ids'].shape[1]} tokens "
      f"(system prompt added automatically)")

plain text (no chat template): 7 tokens
What is the capital of France?

with chat template:
'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\n'

with chat template: 37 tokens (system prompt added automatically)


In [17]:
with torch.no_grad():
    out_plain = model.generate(**plain_ids, max_new_tokens=20, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id)
    out_chat = model.generate(**chat_inputs, max_new_tokens=20, do_sample=False,
                               pad_token_id=tokenizer.pad_token_id)

chat_prompt_len = chat_inputs["input_ids"].shape[1]
plain_prompt_len = plain_ids["input_ids"].shape[1]
text_plain = tokenizer.decode(out_plain[0, plain_prompt_len:], skip_special_tokens=True)
text_chat = tokenizer.decode(out_chat[0, chat_prompt_len:], skip_special_tokens=True)
print(f"without chat template: {text_plain!r}")
print(f"with chat template:    {text_chat!r}")
print(
    "\nThe instruct model was trained on the chat template format - feeding 'plain' "
    "text is not a syntax error, but it is an input from a distribution the model "
    "'formally' never saw during conversation fine-tuning."
)

without chat template: ''
with chat template:    'The capital of France is Paris.'

The instruct model was trained on the chat template format - feeding 'plain' text is not a syntax error, but it is an input from a distribution the model 'formally' never saw during conversation fine-tuning.


In [18]:
# token cost of conversation history - grows with the number of turns, NOT 'free memory'
history = []
turns = [
    "My name is Kasia.",
    "What is my name?",
]
print("=== token cost of a growing conversation history ===")
for turn in turns:
    history.append({"role": "user", "content": turn})
    hist_inputs = tokenizer.apply_chat_template(history, tokenize=True, add_generation_prompt=True,
                                                 return_tensors="pt", return_dict=True)
    hist_len = hist_inputs["input_ids"].shape[1]
    print(f"after adding '{turn}': history = {hist_len} tokens")
    with torch.no_grad():
        out = model.generate(**hist_inputs, max_new_tokens=20, do_sample=False,
                              pad_token_id=tokenizer.pad_token_id)
    reply = tokenizer.decode(out[0, hist_len:], skip_special_tokens=True)
    print(f"  reply: {reply!r}")
    history.append({"role": "assistant", "content": reply})

print(
    "\nThe second turn 'remembers' the name ONLY because the ENTIRE first turn "
    "(user+assistant) was explicitly appended to the input of the second call - the "
    "model itself does not 'remember' anything between generate() calls."
)

=== token cost of a growing conversation history ===
after adding 'My name is Kasia.': history = 36 tokens


  reply: "Kasia, I'm glad you're here to help with your creative writing endeavors. I'm here"
after adding 'What is my name?': history = 72 tokens


  reply: "My name is Kasia, and I'm a creative writing assistant. I'm here to help you"

The second turn 'remembers' the name ONLY because the ENTIRE first turn (user+assistant) was explicitly appended to the input of the second call - the model itself does not 'remember' anything between generate() calls.


## E. KV cache and cost

In [19]:
long_prompt = [{"role": "user", "content":
    "Describe in detail the water cycle, including evaporation, condensation, and precipitation."}]
cache_inputs = tokenizer.apply_chat_template(
    long_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
)
cache_prompt_len = cache_inputs["input_ids"].shape[1]
print(f"prompt: {cache_prompt_len} tokens")

print(f"\n{'n_new':>6} {'use_cache':>10} {'time (s)':>9} {'tok/s':>8}")
for n_new in [16, 32, 64, 128]:
    for use_cache in [True, False]:
        torch.manual_seed(0)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(
                **cache_inputs, min_new_tokens=n_new, max_new_tokens=n_new, do_sample=False,
                use_cache=use_cache, pad_token_id=tokenizer.pad_token_id,
            )
        dt = time.time() - t0
        n_generated = out.shape[1] - cache_prompt_len
        print(f"{n_new:>6} {str(use_cache):>10} {dt:>9.3f} {n_generated/dt:>8.1f}")

print(
    "\nuse_cache=True is clearly faster, with the ADVANTAGE GROWING with generation "
    "length - without a cache, every new step recomputes attention for ALL positions "
    "so far from scratch (step cost grows with position), with a cache - every step "
    "uses the saved K/V (step cost ~constant)."
)

prompt: 45 tokens

 n_new  use_cache  time (s)    tok/s


    16       True     0.147    109.2


    16      False     0.377     42.5


    32       True     0.284    112.8


    32      False     0.883     36.2


    64       True     0.550    116.3


    64      False     2.051     31.2


   128       True     1.089    117.5


   128      False     4.558     28.1

use_cache=True is clearly faster, with the ADVANTAGE GROWING with generation length - without a cache, every new step recomputes attention for ALL positions so far from scratch (step cost grows with position), with a cache - every step uses the saved K/V (step cost ~constant).


In [20]:
# effect of PROMPT length (fixed generation=32 tokens)
short_p = [{"role": "user", "content": "Hello"}]
filler = "The quick brown fox jumps over the lazy dog. " * 40
long_text = "Please summarize in one sentence: " + filler
long_p = [{"role": "user", "content": long_text}]

print(f"{'prompt':40} {'prompt_len':>10} {'time (s)':>9}")
for label, msgs in [("short", short_p), ("long", long_p)]:
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True)
    plen = ids["input_ids"].shape[1]
    torch.manual_seed(0)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**ids, min_new_tokens=32, max_new_tokens=32, do_sample=False,
                              pad_token_id=tokenizer.pad_token_id)
    dt = time.time() - t0
    print(f"{label:40} {plen:>10} {dt:>9.3f}")

print(
    "\nA longer prompt costs more in PREFILL (one pass over the whole prompt), "
    "but for a short generation (32 tokens) DECODE still dominates total time - the "
    "effect of prompt length is real, but not dominant at this scale."
)

prompt                                   prompt_len  time (s)


short                                            31     0.274


long                                            437     0.409

A longer prompt costs more in PREFILL (one pass over the whole prompt), but for a short generation (32 tokens) DECODE still dominates total time - the effect of prompt length is real, but not dominant at this scale.


In [21]:
# KV cache - memory cost: formula vs. actual tensor size
sample_inputs = tokenizer("Hello, how are you today? I am testing the key-value cache.",
                           return_tensors="pt")
with torch.no_grad():
    out = model(**sample_inputs, use_cache=True)
cache = out.past_key_values
seq_len = sample_inputs["input_ids"].shape[1]

n_layers = model.config.num_hidden_layers
n_kv_heads = model.config.num_key_value_heads
head_dim = model.config.head_dim
bytes_per_elem = 4  # float32

formula_bytes = 2 * n_layers * n_kv_heads * head_dim * seq_len * bytes_per_elem


def layer_bytes(layer):
    return (layer.keys.numel() * layer.keys.element_size()
            + layer.values.numel() * layer.values.element_size())


actual_bytes = sum(layer_bytes(layer) for layer in cache.layers)

print(f"model: {n_layers} layers, {model.config.num_attention_heads} Q heads, "
      f"{n_kv_heads} KV heads (GQA!), head_dim={head_dim}")
print(f"sequence: {seq_len} tokens")
print(f"formula:   2 * {n_layers} * {n_kv_heads} * {head_dim} * {seq_len} * {bytes_per_elem} "
      f"= {formula_bytes} bytes = {formula_bytes/1024:.1f} KB")
print(f"actual cache tensor size: {actual_bytes} bytes = {actual_bytes/1024:.1f} KB")
print(f"match (bit-exact)?: {formula_bytes == actual_bytes}")
assert formula_bytes == actual_bytes

print(
    f"\nThis model has {model.config.num_attention_heads} query (Q) heads, but ONLY "
    f"{n_kv_heads} KV heads (grouped-query attention) - the formula MUST use the "
    "number of KV heads, not the number of Q heads, otherwise the memory cost would "
    "be overestimated 3x."
)

model: 30 layers, 9 Q heads, 3 KV heads (GQA!), head_dim=64
sequence: 16 tokens
formula:   2 * 30 * 3 * 64 * 16 * 4 = 737280 bytes = 720.0 KB
actual cache tensor size: 737280 bytes = 720.0 KB
match (bit-exact)?: True

This model has 9 query (Q) heads, but ONLY 3 KV heads (grouped-query attention) - the formula MUST use the number of KV heads, not the number of Q heads, otherwise the memory cost would be overestimated 3x.


## Summary

- Tokenization: the same tokenizer gives ~1.1 tokens/word for English and
  ~3.6 for Polish -- measured, not approximated. Numbers are split into
  individual digits; rare words are split into subwords, not `<UNK>`.
- The manual generation loop (model + argmax + concatenation) gives
  **identical** tokens to `model.generate(do_sample=False)`.
- Temperature changes the **sharpness** of the distribution, not the
  choice of token -- the diversity between decoding strategies is much
  smaller on a concentrated (factual) prompt than on an open-ended
  (creative) one -- diversity is not a universal property of sampling, it
  depends on the distribution for a given prompt.
- Conversation history must be explicitly appended to every call -- the
  model has no memory between `generate()` calls.
- `use_cache=True` is clearly faster than `False`, with the advantage
  **growing** with generation length. KV cache memory cost matches the
  formula $2NH_{kv}d_{\text{head}}T \cdot \text{bytes}$ bit-exactly --
  the model has $9$ query heads, but only $3$ KV heads (GQA), which
  reduces cache memory cost $3\times$ relative to naively counting by
  query heads.

Discussion questions and the bibliography: see [`lecture.en.md`](lecture.en.md).